In [ ]:
import sys

import pandas as pd
from sklearn.metrics import f1_score

sys.path.insert(0, "..")

from src.config import PROJECT_ROOT

#mllm_results_Qwen3-VL-2B_persona_inverted_2026-10-01_14-20.csv
#mllm_results_Qwen3-VL-2B_persona_normal_2026-10-01_14-20.csv
CSV_NAME = "mllm_results_Qwen3-VL-2B_persona_inverted_2026-10-01_14-20.csv"

CSV_FILE = PROJECT_ROOT / "outputs" / "inferences" / CSV_NAME

LABELS = ["left", "right", "equal"]

results = pd.read_csv(CSV_FILE)

human = results["human_answer"].astype(str).str.strip().str.lower()
model = results["model_answer"].astype(str).str.strip().str.lower()

n_total = len(results)

# Valid model predictions
valid = model.isin(LABELS)
n_valid = int(valid.sum())
n_invalid = n_total - n_valid

# Human and model choice counts
human_counts = human.value_counts().reindex(LABELS, fill_value=0)
model_counts = model.where(valid).value_counts().reindex(LABELS, fill_value=0)

# Correct and incorrect predictions
correct = valid & (human == model)
incorrect = valid & (human != model)

n_correct = int(correct.sum())
n_incorrect = int(incorrect.sum())

# Accuracy
accuracy = n_correct / n_total if n_total else 0
accuracy_valid = n_correct / n_valid if n_valid else 0

# F1 scores (valid predictions only)
y_true = human[valid]
y_pred = model[valid]

f1_macro = f1_score(
    y_true, y_pred,
    average="macro",
    labels=LABELS,
    zero_division=0
)

f1_weighted = f1_score(
    y_true, y_pred,
    average="weighted",
    labels=LABELS,
    zero_division=0
)

# Report
print("=" * 55)
print("MODEL EVALUATION REPORT")
print("=" * 55)

print(f"\nFile: {CSV_FILE}")
print(f"Total comparisons: {n_total}")

print("\n1. HUMAN CHOICES")
print("-" * 35)

for label in LABELS:
    print(f"{label.capitalize():<20}: {human_counts[label]:>6}")

print(f"{'Total':<20}: {human_counts.sum():>6}")

print("\n2. MODEL CHOICES")
print("-" * 35)

for label in LABELS:
    print(f"{label.capitalize():<20}: {model_counts[label]:>6}")

print(f"{'Invalid/Error':<20}: {n_invalid:>6}")
print(f"{'Total':<20}: {n_total:>6}")

print("\n3. MODEL vs HUMAN")
print("-" * 35)

print(f"{'Correct':<20}: {n_correct:>6}")
print(f"{'Incorrect':<20}: {n_incorrect:>6}")
print(f"{'Invalid/Error':<20}: {n_invalid:>6}")
print(f"{'Total':<20}: {n_total:>6}")

print(f"\nAccuracy (all comparisons): {accuracy:.4f}")
print(f"Accuracy (valid predictions): {accuracy_valid:.4f}")

print(f"F1 macro (valid predictions): {f1_macro:.4f}")
print(f"F1 weighted (valid predictions): {f1_weighted:.4f}")

print("\n4. ERROR BREAKDOWN")
print("-" * 35)

print(f"Valid but incorrect: {n_incorrect}")
print(f"Invalid/error output: {n_invalid}")
print(f"Total model errors: {n_incorrect + n_invalid}")

print("=" * 55)
